# Stage 2 Control: Random-32 Head LoRA Finetuning

**Goal:** Test whether the causally identified hallucination heads matter by training the same surgical LoRA intervention on a random 32-head control set.

This notebook keeps the important Stage 2 ingredients fixed:
- Same selected training images and held-out eval images
- Same contrastive DPO pairs: COCO ground-truth captions are chosen, Stage 1 model captions are rejected
- Same QLoRA config on Q/K/V projections
- Same number of active head slices: 32
- Same layer footprint by default: random heads are sampled within the same layers and same per-layer counts as the hallucination-head shortlist

The only intended change is the head-selection signal:

```text
causal hallucination heads -> layer-matched random heads
```

This is the reviewer-facing control for:

> Did interpretability-based head selection buy anything over a random set of equally many head slices?

**Outputs are written to separate `stage2_random_heads32_*` files so this will not overwrite the targeted adapter.**

**Prerequisites:** Run/copy the same Stage 1 artifacts used by `stage2_lora.ipynb`:
- `results/final_hallucination_heads.json`
- `cache/screening_state.pkl`
- `cache/selected_imgs.json`
- `coco/val2014_subset/` and `coco/annotations/`


## 0. Install dependencies
**Run once, then Runtime → Restart session. Skip on subsequent runs.**

In [ ]:
# RUN ONCE, then Runtime -> Restart session. Do not run imports before restarting.
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Colab sometimes ships with NumPy 2.x while compiled deps expect NumPy 1.x.
# Pin first, then restart the runtime so the in-memory binary matches the installed wheel.
!pip install -q --no-cache-dir --force-reinstall "numpy==1.26.4"
!pip install -q "transformers>=4.47" "accelerate>=0.33" "tokenizers>=0.21"
!pip install -q peft bitsandbytes
!pip install -q pillow tqdm pycocotools spacy sentencepiece
!python -m spacy download en_core_web_sm -q

print('Done. Now Runtime -> Restart session, then skip this cell and start from imports.')


## 1. Imports and Drive setup
**Start here after session restart.**

In [ ]:
import torch
import numpy as np
import json, os, gc, pickle, random, re
from pathlib import Path
from collections import defaultdict
from PIL import Image
from tqdm.auto import tqdm
import torch.nn.functional as F

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/llava_hallucination_heads'
COCO_DIR = f'{WORK_DIR}/coco'
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)
os.makedirs(f'{WORK_DIR}/cache',   exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    raise RuntimeError(
        'No GPU detected.\n'
        'Fix: Runtime → Change runtime type → GPU (T4 or A100)\n'
        '     Then: Runtime → Disconnect and delete runtime → re-run from Cell 0.'
    )
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 2. Load LLaVA-1.5-7B in 4-bit (QLoRA)

Stage 1 used fp16 *without* 4-bit because quantization breaks attention weight extraction.
Stage 2 only needs **logits** for the DPO loss, so 4-bit is safe and saves ~10 GB VRAM.

In [ ]:
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MODEL_ID = 'llava-hf/llava-1.5-7b-hf'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    device_map={'': 0},
)

# prepare_model_for_kbit_training must run BEFORE get_peft_model:
# - enables gradient checkpointing
# - casts layer norms to fp32 for numerical stability
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print(f'Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 3. Resolve model constants and decoder layers

In [ ]:
# Navigate model internals — handles both old and new transformers layouts
if hasattr(model, 'language_model'):
    lm = model.language_model
elif hasattr(model, 'model') and hasattr(model.model, 'language_model'):
    lm = model.model.language_model
else:
    raise RuntimeError('Cannot find language_model')

if hasattr(lm, 'model') and hasattr(lm.model, 'layers'):
    decoder_layers = lm.model.layers
elif hasattr(lm, 'layers'):
    decoder_layers = lm.layers
else:
    raise RuntimeError('Cannot find decoder layers')

text_cfg         = model.config.text_config
NUM_LAYERS       = text_cfg.num_hidden_layers
NUM_HEADS        = text_cfg.num_attention_heads
HEAD_DIM         = text_cfg.hidden_size // NUM_HEADS
IMAGE_TOKEN_ID   = model.config.image_token_index
vision_cfg       = model.config.vision_config
NUM_IMAGE_TOKENS = (vision_cfg.image_size // vision_cfg.patch_size) ** 2  # 576
PROMPT_TEMPLATE  = 'USER: <image>\nDescribe this image in detail.\nASSISTANT:'

print(f'Layers: {NUM_LAYERS}, Heads/layer: {NUM_HEADS}, head_dim: {HEAD_DIM}')
print(f'Image token id: {IMAGE_TOKEN_ID}, Image tokens: {NUM_IMAGE_TOKENS}')
print(f'Decoder layers: {len(decoder_layers)}')
assert len(decoder_layers) == NUM_LAYERS

## 4. Load Stage 1 artifacts

In [ ]:
from pycocotools.coco import COCO

# --- Original hallucination heads from Stage 1 ---
with open(f'{WORK_DIR}/results/final_hallucination_heads.json') as f:
    original_final_list = json.load(f)

original_heads_by_layer = defaultdict(set)
for h in original_final_list:
    original_heads_by_layer[int(h['layer'])].add(int(h['head']))
original_target_layers = sorted(original_heads_by_layer.keys())

# --- Random-head control configuration ---
RANDOM_HEAD_SEED = 682
MATCH_LAYER_COUNTS = True       # strongest control: same layers and same number of heads per layer
EXCLUDE_ORIGINAL_HEADS = True    # avoid accidentally selecting the causal heads again
CONTROL_NAME = 'random_heads32_layer_matched' if MATCH_LAYER_COUNTS else 'random_heads32_any_layer'

rng = random.Random(RANDOM_HEAD_SEED)

if MATCH_LAYER_COUNTS:
    random_heads_by_layer = defaultdict(set)
    for layer in original_target_layers:
        n_layer_heads = len(original_heads_by_layer[layer])
        forbidden = original_heads_by_layer[layer] if EXCLUDE_ORIGINAL_HEADS else set()
        candidates = [h for h in range(NUM_HEADS) if h not in forbidden]
        if len(candidates) < n_layer_heads:
            raise ValueError(f'Layer {layer} has only {len(candidates)} random candidates for {n_layer_heads} heads.')
        random_heads_by_layer[layer].update(rng.sample(candidates, n_layer_heads))
else:
    original_pairs = {(int(h['layer']), int(h['head'])) for h in original_final_list}
    candidates = [(layer, head) for layer in range(NUM_LAYERS) for head in range(NUM_HEADS)]
    if EXCLUDE_ORIGINAL_HEADS:
        candidates = [p for p in candidates if p not in original_pairs]
    chosen = rng.sample(candidates, len(original_final_list))
    random_heads_by_layer = defaultdict(set)
    for layer, head in chosen:
        random_heads_by_layer[layer].add(head)

# From this point onward, reuse the original variable names used by Stage 2.
# final_list / hal_heads_by_layer now refer to the RANDOM control heads.
hal_heads_by_layer = random_heads_by_layer
target_layers = sorted(hal_heads_by_layer.keys())
final_list = [
    {'layer': int(layer), 'head': int(head), 'control': CONTROL_NAME}
    for layer in target_layers
    for head in sorted(hal_heads_by_layer[layer])
]

random_head_selection = {
    'control_name': CONTROL_NAME,
    'seed': RANDOM_HEAD_SEED,
    'match_layer_counts': MATCH_LAYER_COUNTS,
    'exclude_original_heads': EXCLUDE_ORIGINAL_HEADS,
    'n_heads': len(final_list),
    'n_layers': len(target_layers),
    'target_layers': target_layers,
    'random_heads': final_list,
    'original_heads': original_final_list,
    'overlap_with_original': len(
        {(h['layer'], h['head']) for h in final_list}
        & {(h['layer'], h['head']) for h in original_final_list}
    ),
}

os.makedirs(f'{WORK_DIR}/results', exist_ok=True)
with open(f'{WORK_DIR}/results/stage2_random_heads32_selection.json', 'w') as f:
    json.dump(random_head_selection, f, indent=2)

print(f'Original hallucination heads: {len(original_final_list)} across {len(original_target_layers)} layers')
print(f'Random control heads       : {len(final_list)} across {len(target_layers)} layers')
print(f'Overlap with original heads: {random_head_selection["overlap_with_original"]}')
print(f'Control name               : {CONTROL_NAME}')
print(f'Target layers              : {target_layers}')
for layer in target_layers:
    print(f'  L{layer:02d}: original={sorted(original_heads_by_layer[layer])} random={sorted(hal_heads_by_layer[layer])}')

# --- Per-image cache: model captions + content word labels from Stage 1 ---
CACHE_FILE = f'{WORK_DIR}/cache/screening_state.pkl'
with open(CACHE_FILE, 'rb') as f:
    stage1_state = pickle.load(f)
per_image_cache = stage1_state['per_image_cache']
print(f'Per-image cache: {len(per_image_cache)} entries')

# --- Selected images + ground-truth objects from Stage 1 ---
with open(f'{WORK_DIR}/cache/selected_imgs.json') as f:
    img_meta = json.load(f)
selected_imgs     = img_meta['ids']
img_to_gt_objects = {int(k): set(v) for k, v in img_meta['gt_objects'].items()}

# Rebuild img_id_to_path from COCO annotations
ann_path = f'{COCO_DIR}/annotations/instances_val2014.json'
coco = COCO(ann_path)
img_dir = f'{COCO_DIR}/val2014_subset'
img_meta_coco = coco.loadImgs(selected_imgs)
img_id_to_path = {m['id']: f"{img_dir}/{m['file_name']}" for m in img_meta_coco}

# COCO category names
cat_id_to_name = {c['id']: c['name'].lower() for c in coco.loadCats(coco.getCatIds())}
ALL_COCO_OBJECTS = set(cat_id_to_name.values())

n_exist = sum(1 for p in img_id_to_path.values() if os.path.exists(p))
print(f'Images found: {n_exist}/{len(img_id_to_path)}')


## 5. Build contrastive training pairs

- **Chosen (positive):** COCO ground-truth captions (up to 5 per image)
- **Rejected (negative):** Model's own hallucinated caption from Stage 1
- Only use images where Stage 1 identified at least one hallucinated content word
- **Train/eval split:** last 40 images held out for evaluation

In [ ]:
# Load COCO GT captions (already in annotations zip from Stage 1)
cap_ann_path = f'{COCO_DIR}/annotations/captions_val2014.json'
coco_caps = COCO(cap_ann_path)
print(f'COCO captions loaded: {len(coco_caps.anns)} annotations')

# Fixed train/eval split — last 40 images for eval, first 160 for train
random.seed(42)
eval_img_ids  = set(selected_imgs[-40:])
train_img_ids = set(selected_imgs[:-40])

eval_images     = [img_id for img_id in selected_imgs[-40:] if img_id in img_id_to_path]
eval_gt_objects = [img_to_gt_objects[img_id] for img_id in eval_images]

# Build DPO pairs from train images
dpo_pairs = []
for img_id_raw, cache_info in per_image_cache.items():
    img_id = int(img_id_raw) if isinstance(img_id_raw, str) else img_id_raw
    if img_id not in train_img_ids:
        continue
    img_path = img_id_to_path.get(img_id)
    if not img_path or not os.path.exists(img_path):
        continue

    # Only use images where Stage 1 found hallucinations (rejected caption is meaningful)
    has_hallucination = any(
        cw['is_hallucinated'] for cw in cache_info.get('content_words', [])
    )
    if not has_hallucination:
        continue

    ann_ids = coco_caps.getAnnIds(imgIds=img_id)
    if not ann_ids:
        continue
    gt_captions = [a['caption'].strip() for a in coco_caps.loadAnns(ann_ids)]

    hal_caption = cache_info['caption'].strip()
    for gt_caption in gt_captions:  # up to 5 GT captions per image
        dpo_pairs.append({
            'img_id':   img_id,
            'img_path': img_path,
            'chosen':   gt_caption,   # preferred: grounded GT caption
            'rejected': hal_caption,  # dispreferred: model's own hallucination
        })

random.shuffle(dpo_pairs)
print(f'\nDPO training pairs : {len(dpo_pairs)}')
print(f'Unique train images : {len(set(p["img_id"] for p in dpo_pairs))}')
print(f'Eval images         : {len(eval_images)}')
print(f'\nExample pair:')
print(f'  chosen  : {dpo_pairs[0]["chosen"]}')
print(f'  rejected: {dpo_pairs[0]["rejected"]}')

## 6. Define evaluation helper functions

Self-contained: COCO vocabulary, caption generation, POPE, and CHAIR — no imports from Stage 1.

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

# COCO synonym map and single-word aliases (mirrors Stage 1)
COCO_SYNONYMS = {
    'person':        ['man','woman','people','boy','girl','child','guy','lady','kid',
                      'baby','player','rider','skier','surfer','snowboarder'],
    'car':           ['vehicle','automobile','sedan','suv'],
    'dog':           ['puppy','dogs'],
    'cat':           ['kitten','cats'],
    'tv':            ['television','monitor','screen'],
    'couch':         ['sofa'],
    'cell phone':    ['phone','cellphone','smartphone'],
    'dining table':  ['table','desk'],
    'wine glass':    ['glass'],
    'bicycle':       ['bike'],
    'motorcycle':    ['motorbike'],
    'airplane':      ['plane','jet'],
    'potted plant':  ['plant'],
    'laptop':        ['computer'],
    'refrigerator':  ['fridge'],
    'truck':         ['lorry'],
    'boat':          ['ship','sailboat'],
    'fire hydrant':  ['hydrant'],
    'hot dog':       ['hotdog'],
    'traffic light': ['stoplight'],
    'sports ball':   ['ball','football','soccer ball','basketball'],
    'baseball bat':  ['bat'],
    'tennis racket': ['racket','racquet'],
}
MULTIWORD_ALIASES = {
    'hydrant':'fire hydrant','hotdog':'hot dog','stoplight':'traffic light',
    'bat':'baseball bat','racket':'tennis racket','racquet':'tennis racket',
}
OBJECT_VOCAB = set(ALL_COCO_OBJECTS)
for syns in COCO_SYNONYMS.values():
    OBJECT_VOCAB.update(syns)
OBJECT_VOCAB.update(MULTIWORD_ALIASES.keys())


@torch.no_grad()
def generate_caption(model_obj, image_path, max_new_tokens=80):
    img = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT_TEMPLATE, images=img, return_tensors='pt').to(device, torch.float16)
    inputs['input_ids'] = inputs['input_ids'].long()
    inputs['attention_mask'] = inputs['attention_mask'].long()
    out = model_obj.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             return_dict_in_generate=True, use_cache=True)
    gen_ids = out.sequences[0, inputs['input_ids'].shape[1]:]
    caption = processor.tokenizer.decode(gen_ids, skip_special_tokens=True)
    return caption, gen_ids.cpu()


def find_content_words(gen_ids, gt_objects):
    """Identify nouns in generated caption; label as object/hallucinated. Mirror of Stage 1."""
    full_text = processor.tokenizer.decode(gen_ids, skip_special_tokens=True)
    gt_norm = set(o.lower() for o in gt_objects)
    expanded_gt = set(gt_norm)
    for canonical, syns in COCO_SYNONYMS.items():
        if canonical in gt_norm:
            expanded_gt.update(syns)
    for alias, canonical in MULTIWORD_ALIASES.items():
        if canonical in gt_norm:
            expanded_gt.add(alias)

    doc = nlp(full_text)
    nouns = [tok.text.lower().strip() for tok in doc
             if tok.pos_ in ('NOUN', 'PROPN', 'ADJ') and len(tok.text.strip()) >= 2]
    if not nouns:
        return []

    accumulated = ''
    results = []
    noun_idx = 0
    for tok_i, tid in enumerate(gen_ids):
        if noun_idx >= len(nouns):
            break
        ts = processor.tokenizer.decode([int(tid)], skip_special_tokens=True).lower()
        accumulated += ts
        target = nouns[noun_idx]
        if target in accumulated:
            canonical = MULTIWORD_ALIASES.get(target, target)
            is_object = target in OBJECT_VOCAB
            is_hall   = is_object and (target not in expanded_gt) and (canonical not in expanded_gt)
            results.append({'word': target, 'is_object': is_object, 'is_hallucinated': is_hall})
            cut = accumulated.rfind(target) + len(target)
            accumulated = accumulated[cut:]
            noun_idx += 1
    return results


@torch.no_grad()
def pope_eval(model_obj, images, gt_objects_list, n_per_image=6, seed=0):
    """POPE-style binary evaluation: 'Is there a {obj} in the image?' → yes/no."""
    rng = random.Random(seed)
    coco_cats = list(ALL_COCO_OBJECTS)
    preds, labels = [], []

    model_obj.eval()
    for img_id, gt_set in tqdm(zip(images, gt_objects_list), total=len(images), desc='POPE'):
        img_path = img_id_to_path.get(img_id)
        if not img_path or not os.path.exists(img_path):
            continue
        img = Image.open(img_path).convert('RGB')

        pos_objects = rng.sample(list(gt_set), min(n_per_image // 2, len(gt_set)))
        neg_pool    = [c for c in coco_cats if c not in gt_set]
        neg_objects = rng.sample(neg_pool, min(n_per_image // 2, len(neg_pool)))

        for obj, gt_label in [(o, 1) for o in pos_objects] + [(o, 0) for o in neg_objects]:
            q = f"Is there a {obj} in the image? Please answer yes or no."
            prompt = f"USER: <image>\n{q}\nASSISTANT:"
            inputs = processor(text=prompt, images=img, return_tensors='pt').to(device, torch.float16)
            inputs['input_ids'] = inputs['input_ids'].long()
            inputs['attention_mask'] = inputs['attention_mask'].long()
            out = model_obj.generate(**inputs, max_new_tokens=5, do_sample=False)
            answer = processor.tokenizer.decode(
                out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).lower()
            preds.append(1 if 'yes' in answer else 0)
            labels.append(gt_label)

        torch.cuda.empty_cache()

    preds  = np.array(preds)
    labels = np.array(labels)
    acc    = (preds == labels).mean()
    tp = ((preds == 1) & (labels == 1)).sum()
    fp = ((preds == 1) & (labels == 0)).sum()
    fn = ((preds == 0) & (labels == 1)).sum()
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-9)
    return {'accuracy': float(acc), 'precision': float(prec),
            'recall': float(rec), 'f1': float(f1),
            'yes_rate': float(preds.mean()), 'n': len(preds)}


@torch.no_grad()
def chair_eval(model_obj, images, gt_objects_list):
    """CHAIR-style evaluation on generated captions."""
    model_obj.eval()
    chairs_list, chairi_list = [], []

    for img_id, gt_set in tqdm(zip(images, gt_objects_list), total=len(images), desc='CHAIR'):
        img_path = img_id_to_path.get(img_id)
        if not img_path or not os.path.exists(img_path):
            continue
        caption, gen_ids = generate_caption(model_obj, img_path)
        cw = find_content_words(gen_ids, gt_set)
        obj_words  = [c for c in cw if c['is_object']]
        hall_words = [c for c in cw if c['is_hallucinated']]
        chairs_list.append(1 if hall_words else 0)
        chairi_list.append(len(hall_words) / max(len(obj_words), 1))
        torch.cuda.empty_cache()

    return {'CHAIRs': float(np.mean(chairs_list)),
            'CHAIRi': float(np.mean(chairi_list)),
            'n': len(chairs_list)}


print('Eval functions defined.')

## 7. Baseline evaluation (base model, before LoRA)

Run POPE and CHAIR on the held-out 40 eval images. Takes ~15 minutes.

In [ ]:
print('=== BASELINE EVALUATION (no LoRA) ===')
baseline_pope  = pope_eval(model, eval_images, eval_gt_objects)
baseline_chair = chair_eval(model, eval_images, eval_gt_objects)

baseline_results = {'pope': baseline_pope, 'chair': baseline_chair}
with open(f'{WORK_DIR}/results/stage2_random_heads32_baseline_eval.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print('\nPOPE  :', baseline_pope)
print('CHAIR :', baseline_chair)

## 8. Apply surgical LoRA (PEFT)

`layers_to_transform` restricts LoRA to the layers used by the random-head control. With `MATCH_LAYER_COUNTS=True`, this is the same layer footprint as the causal-head Stage 2 run.

Gradient masking in the next cell further restricts updates to only the 32 random head-slices.


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj'],
    layers_to_transform=target_layers,  # random-head control layers
    lora_dropout=0.05,
    bias='none',
)

model_lora = get_peft_model(model, lora_config)

# Disable any LoRA accidentally added to the vision tower
# (CLIP's self-attn also has q/k/v_proj; layer indices overlap with target_layers)
n_vision_disabled = 0
for name, param in model_lora.named_parameters():
    if ('vision_tower' in name or 'multi_modal_projector' in name) and 'lora_' in name:
        param.requires_grad = False
        n_vision_disabled += 1

model_lora.print_trainable_parameters()
print(f'Vision tower LoRA params disabled: {n_vision_disabled}')

## 9. Register gradient masking hooks

Each `lora_B.weight` has shape `[4096, r]`. Head `h` occupies rows `[h*128 : (h+1)*128]`.
The hook zeros out gradient rows for heads NOT in the random control shortlist, so only the 32 random target head-slices are updated.

In [ ]:
grad_hooks = []
hooks_registered = 0

for name, param in model_lora.named_parameters():
    if 'lora_B' not in name or not param.requires_grad:
        continue
    if 'vision_tower' in name or 'multi_modal_projector' in name:
        continue

    m = re.search(r'layers\.([0-9]+)\.self_attn', name)
    if m is None:
        continue
    layer_idx = int(m.group(1))
    if layer_idx not in hal_heads_by_layer:
        continue

    # Build binary mask: 1 for target head rows, 0 for all others
    # param.shape = [out_features, r] = [4096, 8]
    out_size = param.shape[0]   # 4096 = 32 heads * 128 head_dim
    mask = torch.zeros(out_size, device=device)
    for h in hal_heads_by_layer[layer_idx]:
        mask[h * HEAD_DIM : (h + 1) * HEAD_DIM] = 1.0

    # Closure captures the correct mask for this parameter
    # grad shape: [4096, r]; mask.unsqueeze(-1): [4096, 1] → broadcasts over rank dim
    handle = param.register_hook(lambda g, m=mask: g * m.unsqueeze(-1))
    grad_hooks.append(handle)
    hooks_registered += 1

    active_heads = len(hal_heads_by_layer[layer_idx])
    proj_name = name.split('.')[-4] if '.' in name else '?'
    print(f'  L{layer_idx:02d} {proj_name}: {active_heads}/{NUM_HEADS} heads active')

print(f'\nTotal grad-masking hooks: {hooks_registered}')
print(f'Each hook restricts lora_B updates to {len(final_list)} head-slices total')

In [ ]:
# --- Verify masking: non-target rows should have zero gradient after backward ---
print('Verifying gradient masking ...')
model_lora.train()

_test_img  = Image.open(dpo_pairs[0]['img_path']).convert('RGB')
_test_text = PROMPT_TEMPLATE + dpo_pairs[0]['chosen']
_inputs    = processor(text=_test_text, images=_test_img, return_tensors='pt').to(device, torch.float16)
_inputs['input_ids'] = _inputs['input_ids'].long()
_inputs['attention_mask'] = _inputs['attention_mask'].long()

_out  = model_lora(**_inputs, return_dict=True)
_loss = _out.logits.float().mean()   # dummy loss
_loss.backward()

check_ok = True
for name, param in model_lora.named_parameters():
    if 'lora_B' not in name or param.grad is None:
        continue
    if 'vision_tower' in name:
        continue
    m = re.search(r'layers\.([0-9]+)\.self_attn', name)
    if m is None:
        continue
    layer_idx = int(m.group(1))
    if layer_idx not in hal_heads_by_layer:
        continue

    grad = param.grad  # [4096, r]
    for h in range(NUM_HEADS):
        row_slice = grad[h * HEAD_DIM : (h + 1) * HEAD_DIM]
        norm = row_slice.norm().item()
        is_target = h in hal_heads_by_layer[layer_idx]
        if not is_target and norm > 1e-6:
            print(f'  FAIL: L{layer_idx} head {h} (non-target) grad norm = {norm:.6f}')
            check_ok = False

if check_ok:
    print('PASS: All non-target head rows have zero gradient.')

# Clean up verification pass
model_lora.zero_grad()
del _out, _loss, _inputs, _test_img
torch.cuda.empty_cache()

## 10. Compute PROMPT_LEN

`PROMPT_LEN` is the number of input tokens for the bare prompt (no caption). It is constant across all images because the prompt template is fixed and CLIP always produces 576 patch tokens. We use it to mask the prompt from the DPO loss (labels = -100 for prompt tokens).

In [ ]:
_dummy_img    = Image.open(dpo_pairs[0]['img_path']).convert('RGB')
_prompt_in    = processor(text=PROMPT_TEMPLATE, images=_dummy_img, return_tensors='pt')
_raw_len      = _prompt_in['input_ids'].shape[1]
_n_img_ph     = (_prompt_in['input_ids'] == IMAGE_TOKEN_ID).sum().item()

# New transformers (>=4.47): processor pre-expands to 576; single-placeholder = old behaviour
if _n_img_ph >= NUM_IMAGE_TOKENS:
    PROMPT_LEN = _raw_len
else:
    PROMPT_LEN = _raw_len - 1 + NUM_IMAGE_TOKENS

del _dummy_img, _prompt_in
print(f'PROMPT_LEN = {PROMPT_LEN} (raw input_ids len = {_raw_len}, '
      f'img placeholders = {_n_img_ph})')

## 11. DPO Training Loop

**Loss:**
$$\mathcal{L}_{\text{DPO}} = -\log \sigma\!\left( \beta (\log\pi_\theta(y_w|x) - \log\pi_{\text{ref}}(y_w|x)) - \beta (\log\pi_\theta(y_l|x) - \log\pi_{\text{ref}}(y_l|x)) \right)$$

- $y_w$ = GT caption (chosen), $y_l$ = hallucinated caption (rejected)
- $\pi_{\text{ref}}$ = base model computed via `disable_adapter_layers()` (no extra VRAM)
- Sequence logprob = mean per-token log-prob over caption tokens only

In [ ]:
def make_labels(input_ids, prompt_len):
    """Return labels with -100 for prompt tokens; caption token ids for the rest."""
    labels = torch.full_like(input_ids, -100)
    labels[:, prompt_len:] = input_ids[:, prompt_len:]
    return labels


def sequence_logprob(logits, labels):
    """
    Mean per-token log-prob over labeled positions (next-token prediction).
    logits: [1, T, V] (fp16 or fp32)
    labels: [1, T] with -100 for ignored positions
    Returns: scalar tensor
    """
    shift_logits = logits[:, :-1, :].float()   # [1, T-1, V]
    shift_labels = labels[:, 1:]               # [1, T-1]

    log_probs = F.log_softmax(shift_logits, dim=-1)  # [1, T-1, V]
    mask      = shift_labels != -100
    safe_lbl  = shift_labels.clone()
    safe_lbl[~mask] = 0

    token_lp  = log_probs.gather(2, safe_lbl.unsqueeze(-1)).squeeze(-1)  # [1, T-1]
    token_lp  = token_lp * mask.float()
    return token_lp.sum() / mask.float().sum().clamp_min(1)              # scalar


# --- Hyperparameters ---
BETA          = 0.1
LR            = 1e-4
NUM_EPOCHS    = 3
GRAD_ACCUM    = 4
MAX_GRAD_NORM = 1.0

total_grad_steps = (len(dpo_pairs) * NUM_EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM

optimizer = torch.optim.AdamW(
    [p for p in model_lora.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_grad_steps, eta_min=LR / 10
)

# Checkpoint/resume
STAGE2_CKPT = f'{WORK_DIR}/cache/stage2_checkpoint.json'
start_epoch  = 0
training_log = []
if os.path.exists(STAGE2_CKPT):
    with open(STAGE2_CKPT) as f:
        ckpt = json.load(f)
    start_epoch  = ckpt.get('completed_epochs', 0)
    training_log = ckpt.get('log', [])
    print(f'Resuming from epoch {start_epoch}')

print(f'Training: {len(dpo_pairs)} pairs × {NUM_EPOCHS} epochs')
print(f'Grad updates: {total_grad_steps}  (accum={GRAD_ACCUM})')

In [ ]:
model_lora.train()
optimizer.zero_grad()
global_step = 0

for epoch in range(start_epoch, NUM_EPOCHS):
    random.shuffle(dpo_pairs)
    epoch_loss = 0.0
    n_steps    = 0
    pbar = tqdm(enumerate(dpo_pairs), total=len(dpo_pairs),
                desc=f'Epoch {epoch + 1}/{NUM_EPOCHS}')

    for step, pair in pbar:
        try:
            img = Image.open(pair['img_path']).convert('RGB')

            # --- Chosen (GT caption) ---
            chosen_in = processor(
                text=PROMPT_TEMPLATE + pair['chosen'], images=img,
                return_tensors='pt').to(device, torch.float16)
            chosen_in['input_ids']      = chosen_in['input_ids'].long()
            chosen_in['attention_mask'] = chosen_in['attention_mask'].long()
            chosen_labels = make_labels(chosen_in['input_ids'], PROMPT_LEN).to(device)

            # --- Rejected (hallucinated caption) ---
            rejected_in = processor(
                text=PROMPT_TEMPLATE + pair['rejected'], images=img,
                return_tensors='pt').to(device, torch.float16)
            rejected_in['input_ids']      = rejected_in['input_ids'].long()
            rejected_in['attention_mask'] = rejected_in['attention_mask'].long()
            rejected_labels = make_labels(rejected_in['input_ids'], PROMPT_LEN).to(device)

            # --- Reference log-probs (LoRA disabled, no gradient) ---
            model_lora.disable_adapter_layers()
            with torch.no_grad():
                ref_lp_chosen   = sequence_logprob(
                    model_lora(**chosen_in,   return_dict=True).logits, chosen_labels)
                ref_lp_rejected = sequence_logprob(
                    model_lora(**rejected_in, return_dict=True).logits, rejected_labels)
            model_lora.enable_adapter_layers()

            # --- Policy log-probs (LoRA enabled, with gradient) ---
            lp_chosen   = sequence_logprob(
                model_lora(**chosen_in,   return_dict=True).logits, chosen_labels)
            lp_rejected = sequence_logprob(
                model_lora(**rejected_in, return_dict=True).logits, rejected_labels)

            # --- DPO loss ---
            delta_chosen   = BETA * (lp_chosen   - ref_lp_chosen.detach())
            delta_rejected = BETA * (lp_rejected - ref_lp_rejected.detach())
            loss = -F.logsigmoid(delta_chosen - delta_rejected)

            (loss / GRAD_ACCUM).backward()
            epoch_loss += loss.item()
            n_steps    += 1

            if (step + 1) % GRAD_ACCUM == 0 or step == len(dpo_pairs) - 1:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model_lora.parameters() if p.requires_grad],
                    MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % 20 == 0:
                    pbar.set_postfix({
                        'loss': f'{epoch_loss / n_steps:.4f}',
                        'lr':   f'{scheduler.get_last_lr()[0]:.2e}',
                    })

            del chosen_in, rejected_in, chosen_labels, rejected_labels
            torch.cuda.empty_cache()

        except Exception as e:
            print(f'\nStep {step} failed: {type(e).__name__}: {e}')
            optimizer.zero_grad()
            torch.cuda.empty_cache()

    avg_loss = epoch_loss / max(n_steps, 1)
    training_log.append({'epoch': epoch + 1, 'loss': avg_loss, 'steps': n_steps})
    print(f'Epoch {epoch + 1}: avg_loss={avg_loss:.4f}  steps={n_steps}')

    with open(STAGE2_CKPT, 'w') as f:
        json.dump({'completed_epochs': epoch + 1, 'log': training_log}, f, indent=2)

print('\nTraining complete.')
print('Training log:', training_log)

## 12. Post-training evaluation (LoRA enabled)

In [ ]:
model_lora.enable_adapter_layers()

print('=== POST-TRAINING EVALUATION (random-32-head surgical LoRA control) ===')
lora_pope  = pope_eval(model_lora, eval_images, eval_gt_objects)
lora_chair = chair_eval(model_lora, eval_images, eval_gt_objects)

lora_results = {'pope': lora_pope, 'chair': lora_chair}
with open(f'{WORK_DIR}/results/stage2_random_heads32_lora_eval.json', 'w') as f:
    json.dump(lora_results, f, indent=2)

print('\nPOPE  :', lora_pope)
print('CHAIR :', lora_chair)

## 13. Comparison table

In [ ]:
b = baseline_results
l = lora_results

def d(after, before, higher_is_better=True):
    delta = after - before
    sign  = '+' if delta >= 0 else ''
    arrow = '↑' if (delta > 0) == higher_is_better else ('↓' if delta != 0 else '=')
    return f'{sign}{delta:.4f} {arrow}'

rows = [
    ('POPE Accuracy',  b['pope']['accuracy'],  l['pope']['accuracy'],  True),
    ('POPE F1',        b['pope']['f1'],         l['pope']['f1'],         True),
    ('POPE Precision', b['pope']['precision'],  l['pope']['precision'],  True),
    ('POPE Recall',    b['pope']['recall'],     l['pope']['recall'],     True),
    ('POPE Yes-rate',  b['pope']['yes_rate'],   l['pope']['yes_rate'],   None),
    ('CHAIRs',         b['chair']['CHAIRs'],    l['chair']['CHAIRs'],    False),
    ('CHAIRi',         b['chair']['CHAIRi'],    l['chair']['CHAIRi'],    False),
]

print('=' * 62)
print(f'{"Metric":<20} {"Baseline":>10} {"LoRA":>10} {"Delta":>16}')
print('-' * 62)
for name, bv, lv, hib in rows:
    delta_str = d(lv, bv, hib) if hib is not None else f'{lv - bv:+.4f}'
    print(f'{name:<20} {bv:>10.4f} {lv:>10.4f} {delta_str:>16}')
print('=' * 62)
print(f'Training pairs used : {len(dpo_pairs)}')
print(f'Epochs              : {NUM_EPOCHS}')
print(f'Random heads (LoRA) : {len(final_list)} / 1024 ({100*len(final_list)/1024:.1f}%)')
print(f'Control name        : {CONTROL_NAME}')

## 14. Save LoRA adapter

In [ ]:
adapter_dir = f'{WORK_DIR}/results/stage2_random_heads32_lora_adapter'
model_lora.save_pretrained(adapter_dir)
print(f'LoRA adapter saved to: {adapter_dir}')
print('Files:')
for f_name in os.listdir(adapter_dir):
    fpath = os.path.join(adapter_dir, f_name)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {f_name:<40} {size_mb:.1f} MB')

# Save training log
log_path = f'{WORK_DIR}/results/stage2_random_heads32_training_log.json'
with open(log_path, 'w') as f:
    json.dump({'hyperparams': {'BETA': BETA, 'LR': LR, 'NUM_EPOCHS': NUM_EPOCHS,
                               'GRAD_ACCUM': GRAD_ACCUM, 'lora_rank': 8, 'lora_alpha': 16,
                               'n_target_heads': len(final_list), 'n_target_layers': len(target_layers),
                               'control_name': CONTROL_NAME, 'random_head_seed': RANDOM_HEAD_SEED,
                               'match_layer_counts': MATCH_LAYER_COUNTS,
                               'exclude_original_heads': EXCLUDE_ORIGINAL_HEADS},
               'random_head_selection': random_head_selection,
               'training_log': training_log,
               'baseline': baseline_results,
               'lora_eval': lora_results}, f, indent=2)
print(f'Training log saved to: {log_path}')

## Stage 2 Random-32-Head Control Complete

### What was done
- Loaded LLaVA-1.5-7B in **4-bit (QLoRA)**.
- Used the same Stage 2 DPO training pairs and held-out eval split.
- Applied the same **LoRA (r=8, alpha=16)** configuration to Q/K/V projections.
- Replaced the causal hallucination-head shortlist with a **random 32-head control set**.
- By default, random heads are **layer-count matched** to the original causal shortlist, preserving the same LoRA layer footprint.
- Registered gradient masking hooks so only the random head-slices receive LoRA updates.

### Outputs saved to Drive
| File | Description |
|------|-------------|
| `results/stage2_random_heads32_selection.json` | Exact random head selection and original-head overlap |
| `results/stage2_random_heads32_lora_adapter/` | Random-head LoRA adapter weights |
| `results/stage2_random_heads32_baseline_eval.json` | Baseline eval before random-head LoRA |
| `results/stage2_random_heads32_lora_eval.json` | Random-head LoRA eval |
| `results/stage2_random_heads32_training_log.json` | Epoch losses, hyperparams, and control metadata |

### How to use in the paper
Compare:
- Targeted hallucination-head LoRA
- Random-32-head LoRA control
- Stage 4 targeted LoRA + grounding

If targeted LoRA beats this random-head control at comparable length, the actionability claim is stronger. If they tie, the honest result is that DPO/conservatism is doing most of the behavioral work and the head selection is not yet validated.
